# WMS Colab Companion v2

Web Media Studio用のColab + Gradio Companionです。
WMSから受け取ったURLを、Colab上のyt-dlp + Deno + FFmpegでMP3 / M4A / WAVへLocalizeします。

**自分が権利を持つ、または保存・変換の許可を得ているコンテンツだけに使用してください。**
Colabランタイムと `gradio.live` URLは一時的です。


In [ ]:
#@title STEP 1 — Companionを準備 { display-mode: "form" }
WMS_REF = "main" #@param {type:"string"}

import pathlib
import shutil
import subprocess
import sys
import urllib.request
import zipfile

GRADIO_VERSION = "6.26.0"
YTDLP_VERSION = "2026.8.19"
DENO_VERSION = "2.9.6"
ROOT = pathlib.Path("/content/wms-colab-companion-v2")
ROOT.mkdir(parents=True, exist_ok=True)

print(f"[1/4] Gradio {GRADIO_VERSION} + yt-dlp {YTDLP_VERSION} を準備しています…")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"gradio=={GRADIO_VERSION}", f"yt-dlp[default]=={YTDLP_VERSION}"], check=True)

print(f"[2/4] Deno {DENO_VERSION} を確認しています…")
deno_path = pathlib.Path("/usr/local/bin/deno")
needs_deno = True
if deno_path.exists():
    try:
        current = subprocess.check_output([str(deno_path), "--version"], text=True).splitlines()[0]
        needs_deno = DENO_VERSION not in current
    except Exception:
        needs_deno = True
if needs_deno:
    archive = pathlib.Path(f"/tmp/deno-{DENO_VERSION}.zip")
    urllib.request.urlretrieve(f"https://github.com/denoland/deno/releases/download/v{DENO_VERSION}/deno-x86_64-unknown-linux-gnu.zip", archive)
    with zipfile.ZipFile(archive) as zf:
        zf.extract("deno", "/usr/local/bin")
    deno_path.chmod(0o755)

print("[3/4] FFmpegを確認しています…")
if shutil.which("ffmpeg") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)

print("[4/4] WMS Companionアプリを取得しています…")
base = f"https://raw.githubusercontent.com/goroyattemiyo/web-media-studio/{WMS_REF}/colab_companion"
for name in ["wms_colab_companion.py", "wms_colab_companion.css"]:
    target = ROOT / name
    urllib.request.urlretrieve(f"{base}/{name}", target)
    print(f"取得: {target}")

print("準備完了。STEP 2を実行してください。")


## STEP 2 — Companionを起動

下のセルを実行すると `https://xxxxx.gradio.live` のURLが表示されます。
このセルはCompanion利用中、実行中のままで正常です。


In [ ]:
%cd /content/wms-colab-companion-v2
!python -u wms_colab_companion.py --server-name 0.0.0.0 --server-port 7860 --share
